# RAG Safety Experiments
Additional experiments related to An et al., 2025.

**Runtime requirement:** Google Colab A100 High-RAM (80 GB GPU, 167 GB system RAM).

**Two loading modes (set `USE_4BIT` in Cell 1):**
- `USE_4BIT = False` — bfloat16, sequential loading. Command-R is unloaded before Llama Guard is loaded. Better generation quality. Peak GPU RAM ~67 GB.
- `USE_4BIT = True`  — 4-bit NF4 quantization, all models in memory simultaneously. Peak GPU RAM ~23 GB. Slightly lower quality.

**Pipeline overview:**
1. Build (or reload) a FAISS vector index from a shuffled sample of the English Wikipedia bge-m3 dataset.
2. For each question in your input file, generate a RAG response (retrieved context + Command-R) and a no-RAG response (Command-R only).
3. Judge both responses with Llama Guard 3 8B.
4. All intermediate results are checkpointed to disk.

## Cell Group 0 — Installation

In [ ]:
# Colab ships with torch; reinstalling it wastes time and can break CUDA.
# Only the packages below need installing.
!pip install -q \
    "transformers>=4.44.0" \
    "accelerate>=0.30.0" \
    "bitsandbytes>=0.43.1" \
    "faiss-cpu>=1.8.0" \
    "datasets>=2.20.0" \
    "huggingface_hub>=0.23.0" \
    "sentence-transformers>=3.0.0" \
    "pandas>=2.2.0" \
    "numpy>=1.26.0" \
    "tqdm>=4.66.0"
print("Installation complete. If this is the first run, go to Runtime → Restart session, then continue from the next cell.")

In [ ]:
import importlib, sys

REQUIRED = {
    "torch":               "2.3.0",
    "transformers":        "4.44.0",
    "accelerate":          "0.30.0",
    "bitsandbytes":        "0.43.1",
    "faiss":               "1.8.0",
    "datasets":            "2.20.0",
    "sentence_transformers": "3.0.0",
    "pandas":              "2.2.0",
    "numpy":               "1.26.0",
}

from packaging.version import Version
ok = True
for pkg, min_ver in REQUIRED.items():
    try:
        mod = importlib.import_module(pkg)
        ver = getattr(mod, "__version__", "0")
        status = "OK" if Version(ver) >= Version(min_ver) else "OUTDATED"
        if status == "OUTDATED":
            ok = False
        print(f"  {status:8s} {pkg} {ver} (need >={min_ver})")
    except ImportError:
        print(f"  MISSING  {pkg}")
        ok = False

import torch
cuda_ok = torch.cuda.is_available()
gpu_name = torch.cuda.get_device_name(0) if cuda_ok else "none"
gpu_mem  = torch.cuda.get_device_properties(0).total_memory / 1e9 if cuda_ok else 0
print(f"\n  GPU: {gpu_name} ({gpu_mem:.1f} GB)")
if not cuda_ok:
    print("  WARNING: CUDA not available — make sure you selected the A100 runtime.")
if gpu_mem < 70:
    print(f"  WARNING: GPU has {gpu_mem:.1f} GB. bfloat16 mode needs ~67 GB; switch to USE_4BIT=True if below that.")
if not ok:
    print("\n  Some packages are outdated or missing — re-run the install cell and restart the runtime.")
else:
    print("\n  All packages OK.")

## Cell Group 1 — Imports & Configuration

In [ ]:
# ── Loading mode ─────────────────────────────────────────────────────────────
# False → bfloat16, sequential (unload Command-R before loading Llama Guard)
# True  → 4-bit NF4, all three models in GPU memory simultaneously
USE_4BIT: bool = False

# ── HuggingFace credentials ───────────────────────────────────────────────────
# Required for Llama Guard (gated model). Run: huggingface-cli login
# or set HF_TOKEN here and it will be passed to from_pretrained().
HF_TOKEN: str = ""   # leave empty if you have already run huggingface-cli login

# ── Model IDs ────────────────────────────────────────────────────────────────
COMMAND_R_ID    = "CohereLabs/c4ai-command-r-08-2024"
LLAMA_GUARD_ID  = "meta-llama/Llama-Guard-3-8B"
BGE_M3_ID       = "BAAI/bge-m3"

# ── Wikipedia RAG store ───────────────────────────────────────────────────────
WIKI_DATASET_ID = "Upstash/wikipedia-2024-06-bge-m3"
WIKI_LANG       = "en"
SUBSET_SIZE     = 500_000   # number of paragraphs to index; adjust as needed
SHUFFLE_SEED    = 42
SHUFFLE_BUFFER  = 50_000    # larger = better randomness, more RAM during build
EMBEDDING_DIM   = 1024      # bge-m3 dense embedding dimension

# ── Retrieval & generation ────────────────────────────────────────────────────
TOP_K           = 5         # retrieved passages per query
MAX_NEW_TOKENS  = 256
TEMPERATURE     = 0.3

# ── File paths ────────────────────────────────────────────────────────────────
FAISS_INDEX_PATH    = "wiki_faiss.index"
CORPUS_TEXTS_PATH   = "wiki_texts.parquet"
INPUT_FILE          = "questions.csv"      # must have a 'question' column
RAG_OUTPUT_FILE     = "results_rag.csv"
NO_RAG_OUTPUT_FILE  = "results_no_rag.csv"
JUDGE_OUTPUT_FILE   = "results_judged.csv"

CHECKPOINT_EVERY    = 50    # flush results to disk every N rows

print(f"Loading mode  : {'4-bit NF4 (simultaneous)' if USE_4BIT else 'bfloat16 (sequential)'}")
print(f"Subset size   : {SUBSET_SIZE:,} paragraphs")
print(f"Top-K         : {TOP_K}")
print(f"Max new tokens: {MAX_NEW_TOKENS}")
print(f"Temperature   : {TEMPERATURE}")

In [ ]:
import gc
import itertools
import os
import warnings
warnings.filterwarnings("ignore")

import faiss
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
TORCH_DTYPE = torch.bfloat16
print(f"Device: {DEVICE}")

## Cell Group 2 — Load Command-R

In [ ]:
def get_bnb_config() -> BitsAndBytesConfig | None:
    """Return a 4-bit NF4 config when USE_4BIT is True, else None."""
    if not USE_4BIT:
        return None
    return BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
    )

In [ ]:
token_kwarg = {"token": HF_TOKEN} if HF_TOKEN else {}

print(f"Loading Command-R tokenizer from {COMMAND_R_ID} …")
cr_tokenizer = AutoTokenizer.from_pretrained(COMMAND_R_ID, **token_kwarg)

bnb_config = get_bnb_config()
load_kwargs = dict(
    device_map="auto",
    **token_kwarg,
)
if bnb_config is not None:
    load_kwargs["quantization_config"] = bnb_config
else:
    load_kwargs["torch_dtype"] = TORCH_DTYPE

print(f"Loading Command-R model ({'4-bit NF4' if bnb_config else 'bfloat16'}) …")
print("  This will download ~64 GB on first run and may take 20-40 minutes.")
cr_model = AutoModelForCausalLM.from_pretrained(COMMAND_R_ID, **load_kwargs)
cr_model.eval()

mem = torch.cuda.memory_allocated() / 1e9
print(f"GPU memory in use after Command-R load: {mem:.1f} GB")

In [ ]:
# Quick smoke test — single short prompt.
_test_inputs = cr_tokenizer("Hello, who are you?", return_tensors="pt").to(DEVICE)
with torch.no_grad():
    _test_out = cr_model.generate(**_test_inputs, max_new_tokens=30)
_test_decoded = cr_tokenizer.decode(_test_out[0], skip_special_tokens=True)
print("Smoke test output:", _test_decoded)

## Cell Group 3 — Build or Load RAG Store

In [ ]:
print(f"Loading bge-m3 query encoder from {BGE_M3_ID} …")
encoder = SentenceTransformer(BGE_M3_ID, device=DEVICE)
encoder.eval()
print("bge-m3 loaded.")

In [ ]:
if os.path.exists(FAISS_INDEX_PATH) and os.path.exists(CORPUS_TEXTS_PATH):
    print("Found existing index and corpus — loading from disk.")
    faiss_index = faiss.read_index(FAISS_INDEX_PATH)
    corpus_df   = pd.read_parquet(CORPUS_TEXTS_PATH)
    corpus_texts = corpus_df["text"].tolist()
    print(f"Loaded {faiss_index.ntotal:,} vectors, {len(corpus_texts):,} texts.")
else:
    print(f"Building RAG index from {SUBSET_SIZE:,} sampled Wikipedia paragraphs …")
    print(f"  Dataset : {WIKI_DATASET_ID} / {WIKI_LANG}")
    print(f"  Shuffle seed={SHUFFLE_SEED}, buffer={SHUFFLE_BUFFER:,}")

    ds = load_dataset(WIKI_DATASET_ID, WIKI_LANG, split="train", streaming=True)
    ds = ds.shuffle(seed=SHUFFLE_SEED, buffer_size=SHUFFLE_BUFFER)

    corpus_texts: list[str] = []
    embeddings_list: list[np.ndarray] = []

    for row in tqdm(itertools.islice(ds, SUBSET_SIZE), total=SUBSET_SIZE, desc="Streaming"):
        corpus_texts.append(row["text"])
        embeddings_list.append(row["embedding"])

    actual_size = len(corpus_texts)
    if actual_size < SUBSET_SIZE:
        print(f"  WARNING: only {actual_size:,} rows available (SUBSET_SIZE={SUBSET_SIZE:,}).")

    # Inspect a sample row to confirm text format.
    print("\nSample paragraph (first row):")
    print(corpus_texts[0][:500])

    embeddings = np.array(embeddings_list, dtype=np.float32)  # (N, 1024)
    faiss.normalize_L2(embeddings)                             # cosine → inner product

    faiss_index = faiss.IndexFlatIP(EMBEDDING_DIM)
    faiss_index.add(embeddings)

    faiss.write_index(faiss_index, FAISS_INDEX_PATH)
    pd.DataFrame({"text": corpus_texts}).to_parquet(CORPUS_TEXTS_PATH, index=False)

    print(f"\nIndex saved  → {FAISS_INDEX_PATH}")
    print(f"Corpus saved → {CORPUS_TEXTS_PATH}")
    print(f"Total indexed: {faiss_index.ntotal:,} paragraphs")

## Cell Group 4 — Retrieval & Generation Functions

In [ ]:
def retrieve(query: str, top_k: int = TOP_K) -> list[str]:
    """Return the top-k corpus passages most relevant to query."""
    q_emb = encoder.encode([query], normalize_embeddings=True).astype(np.float32)
    _, indices = faiss_index.search(q_emb, top_k)
    return [corpus_texts[i] for i in indices[0] if i >= 0]

In [ ]:
@torch.no_grad()
def generate_with_rag(
    query: str,
    top_k: int = TOP_K,
    max_new_tokens: int = MAX_NEW_TOKENS,
    temperature: float = TEMPERATURE,
    citation_mode: str = "accurate",
) -> dict:
    """
    Retrieve top-k passages and generate a grounded response.

    Uses Command-R's grounded generation template as recommended by CohereLabs.
    Returns a dict with keys: question, sources, response.
    """
    sources = retrieve(query, top_k)
    documents = [{"title": f"Source {i + 1}", "text": text} for i, text in enumerate(sources)]

    conversation = [{"role": "user", "content": query}]
    prompt = cr_tokenizer.apply_grounded_generation_template(
        conversation,
        documents=documents,
        citation_mode=citation_mode,
        tokenize=False,
        add_generation_prompt=True,
    )
    input_ids = cr_tokenizer(
        prompt, return_tensors="pt", truncation=True, max_length=4096
    ).input_ids.to(DEVICE)

    gen_tokens = cr_model.generate(
        input_ids,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=temperature,
        pad_token_id=cr_tokenizer.eos_token_id,
    )
    new_ids  = gen_tokens[0][input_ids.shape[-1]:]
    response = cr_tokenizer.decode(new_ids, skip_special_tokens=True).strip()

    return {"question": query, "sources": sources, "response": response}

In [ ]:
@torch.no_grad()
def generate_without_rag(
    query: str,
    max_new_tokens: int = MAX_NEW_TOKENS,
    temperature: float = TEMPERATURE,
) -> dict:
    """
    Generate a response using only the model's parametric knowledge.

    Uses Command-R's chat template as recommended by CohereLabs.
    Returns a dict with keys: question, response.
    """
    messages = [{"role": "user", "content": query}]
    input_ids = cr_tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to(DEVICE)

    gen_tokens = cr_model.generate(
        input_ids,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=temperature,
        pad_token_id=cr_tokenizer.eos_token_id,
    )
    new_ids  = gen_tokens[0][input_ids.shape[-1]:]
    response = cr_tokenizer.decode(new_ids, skip_special_tokens=True).strip()

    return {"question": query, "response": response}

## Cell Group 5 — Individual Query Testing

In [ ]:
# ── Edit the question below and re-run this cell to test RAG generation. ─────
_test_question = "Who wrote the novel 1984?"

_rag_result = generate_with_rag(_test_question)

print("=" * 60)
print("QUESTION:", _rag_result["question"])
print("-" * 60)
for i, src in enumerate(_rag_result["sources"], 1):
    print(f"[Source {i}] {src[:200]}")
print("-" * 60)
print("RESPONSE:", _rag_result["response"])

In [ ]:
# ── Edit the question below and re-run this cell to test no-RAG generation. ──
_test_question = "Who wrote the novel 1984?"

_no_rag_result = generate_without_rag(_test_question)

print("=" * 60)
print("QUESTION:", _no_rag_result["question"])
print("-" * 60)
print("RESPONSE:", _no_rag_result["response"])

## Cell Group 6 — Batch Inference Loop

Reads questions from `INPUT_FILE` (CSV with a `question` column).  
Results are saved to `RAG_OUTPUT_FILE` and `NO_RAG_OUTPUT_FILE` every `CHECKPOINT_EVERY` rows.  
Re-running the cell resumes from the last completed question.

In [ ]:
def _append_rows(path: str, rows: list[dict], write_header: bool) -> None:
    pd.DataFrame(rows).to_csv(
        path, mode="a", header=write_header, index=False
    )


def run_batch_inference(
    input_file: str = INPUT_FILE,
    rag_output: str = RAG_OUTPUT_FILE,
    no_rag_output: str = NO_RAG_OUTPUT_FILE,
    top_k: int = TOP_K,
    max_new_tokens: int = MAX_NEW_TOKENS,
    temperature: float = TEMPERATURE,
    checkpoint_every: int = CHECKPOINT_EVERY,
) -> None:
    questions_df = pd.read_csv(input_file)
    if "question" not in questions_df.columns:
        raise ValueError(f"{input_file!r} must have a 'question' column.")

    # Resume: collect already-completed questions from both output files.
    completed: set[str] = set()
    for path in (rag_output, no_rag_output):
        if os.path.exists(path):
            completed.update(pd.read_csv(path)["question"].tolist())

    pending = questions_df[~questions_df["question"].isin(completed)]
    print(f"Total questions : {len(questions_df):,}")
    print(f"Already done    : {len(completed):,}")
    print(f"Remaining       : {len(pending):,}")

    rag_buf, no_rag_buf = [], []
    rag_header    = not os.path.exists(rag_output)
    no_rag_header = not os.path.exists(no_rag_output)

    for i, row in enumerate(
        tqdm(pending.itertuples(index=False), total=len(pending), desc="Inference"), 1
    ):
        q = row.question
        rag_buf.append(generate_with_rag(q, top_k=top_k, max_new_tokens=max_new_tokens, temperature=temperature))
        no_rag_buf.append(generate_without_rag(q, max_new_tokens=max_new_tokens, temperature=temperature))

        if i % checkpoint_every == 0:
            _append_rows(rag_output, rag_buf, rag_header)
            _append_rows(no_rag_output, no_rag_buf, no_rag_header)
            rag_header = no_rag_header = False
            rag_buf.clear()
            no_rag_buf.clear()

    # Flush remainder.
    if rag_buf:
        _append_rows(rag_output, rag_buf, rag_header)
        _append_rows(no_rag_output, no_rag_buf, no_rag_header)

    print(f"Done. Results saved to {rag_output!r} and {no_rag_output!r}.")


# Uncomment to run:
# run_batch_inference()

## Cell Group 7 — Transition: Unload Command-R / Load Llama Guard

- **`USE_4BIT = True`**: all models are already in memory — this cell is a no-op.
- **`USE_4BIT = False`**: this cell frees Command-R from GPU memory before loading Llama Guard.

**Run `run_batch_inference()` above and confirm results are saved before continuing.**

In [ ]:
if USE_4BIT:
    print("USE_4BIT=True — all models remain in memory. No transition needed.")
else:
    print("USE_4BIT=False — unloading Command-R to free GPU memory …")
    del cr_model
    del cr_tokenizer
    gc.collect()
    torch.cuda.empty_cache()
    mem = torch.cuda.memory_allocated() / 1e9
    print(f"GPU memory after unload: {mem:.1f} GB")
    print("Ready to load Llama Guard.")

## Cell Group 8 — Load Llama Guard 3 8B

In [ ]:
token_kwarg = {"token": HF_TOKEN} if HF_TOKEN else {}

print(f"Loading Llama Guard tokenizer from {LLAMA_GUARD_ID} …")
guard_tokenizer = AutoTokenizer.from_pretrained(LLAMA_GUARD_ID, **token_kwarg)

bnb_config = get_bnb_config()
load_kwargs = dict(device_map="auto", **token_kwarg)
if bnb_config is not None:
    load_kwargs["quantization_config"] = bnb_config
else:
    load_kwargs["torch_dtype"] = TORCH_DTYPE

print(f"Loading Llama Guard model ({'4-bit NF4' if bnb_config else 'bfloat16'}) …")
guard_model = AutoModelForCausalLM.from_pretrained(LLAMA_GUARD_ID, **load_kwargs)
guard_model.eval()

mem = torch.cuda.memory_allocated() / 1e9
print(f"GPU memory in use after Llama Guard load: {mem:.1f} GB")

## Cell Group 9 — Safety Judge Functions

In [ ]:
@torch.no_grad()
def judge_safety(
    user_message: str,
    assistant_response: str,
    max_new_tokens: int = 30,
) -> dict:
    """
    Judge whether an (user_message, assistant_response) pair is safe.

    Uses Llama Guard 3's built-in chat template.
    Returns a dict with keys: verdict ('safe' | 'unsafe'), category (str | None), raw (str).
    """
    conversation = [
        {"role": "user",      "content": user_message},
        {"role": "assistant", "content": assistant_response},
    ]
    prompt = guard_tokenizer.apply_chat_template(
        conversation,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = guard_tokenizer(prompt, return_tensors="pt").to(DEVICE)
    output_ids = guard_model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=guard_tokenizer.eos_token_id,
    )
    new_ids = output_ids[0][inputs["input_ids"].shape[-1]:]
    raw     = guard_tokenizer.decode(new_ids, skip_special_tokens=True).strip()

    # Llama Guard output is 'safe' or 'unsafe\n<category>'.
    lines    = raw.splitlines()
    verdict  = lines[0].strip().lower() if lines else "unknown"
    category = lines[1].strip() if verdict == "unsafe" and len(lines) > 1 else None

    return {"verdict": verdict, "category": category, "raw": raw}

## Cell Group 10 — Individual Judge Testing

In [ ]:
# ── Edit and re-run to test the judge on a single pair. ──────────────────────
_test_user = "Who wrote the novel 1984?"
_test_resp = "The novel 1984 was written by George Orwell and published in 1949."

_judge_result = judge_safety(_test_user, _test_resp)
print("Verdict  :", _judge_result["verdict"])
print("Category :", _judge_result["category"])
print("Raw      :", _judge_result["raw"])

## Cell Group 11 — Batch Safety Judging Loop

Reads `RAG_OUTPUT_FILE` and `NO_RAG_OUTPUT_FILE`, judges every (question, response) pair,  
and writes results to `JUDGE_OUTPUT_FILE`. Re-running resumes from the last completed row.

In [ ]:
def run_batch_judging(
    rag_input: str = RAG_OUTPUT_FILE,
    no_rag_input: str = NO_RAG_OUTPUT_FILE,
    judge_output: str = JUDGE_OUTPUT_FILE,
    checkpoint_every: int = CHECKPOINT_EVERY,
) -> None:
    rag_df    = pd.read_csv(rag_input)
    no_rag_df = pd.read_csv(no_rag_input)

    # Merge on question so columns are aligned.
    merged = rag_df[["question", "response"]].rename(
        columns={"response": "rag_response"}
    ).merge(
        no_rag_df[["question", "response"]].rename(columns={"response": "no_rag_response"}),
        on="question",
        how="inner",
    )

    # Resume: skip questions already in the output file.
    completed: set[str] = set()
    if os.path.exists(judge_output):
        completed = set(pd.read_csv(judge_output)["question"].tolist())

    pending = merged[~merged["question"].isin(completed)]
    print(f"Total pairs : {len(merged):,}")
    print(f"Already done: {len(completed):,}")
    print(f"Remaining   : {len(pending):,}")

    buf = []
    write_header = not os.path.exists(judge_output)

    for i, row in enumerate(
        tqdm(pending.itertuples(index=False), total=len(pending), desc="Judging"), 1
    ):
        rag_judge    = judge_safety(row.question, row.rag_response)
        no_rag_judge = judge_safety(row.question, row.no_rag_response)

        buf.append({
            "question":        row.question,
            "rag_response":    row.rag_response,
            "no_rag_response": row.no_rag_response,
            "rag_verdict":     rag_judge["verdict"],
            "rag_category":    rag_judge["category"],
            "no_rag_verdict":  no_rag_judge["verdict"],
            "no_rag_category": no_rag_judge["category"],
        })

        if i % checkpoint_every == 0:
            pd.DataFrame(buf).to_csv(judge_output, mode="a", header=write_header, index=False)
            write_header = False
            buf.clear()

    if buf:
        pd.DataFrame(buf).to_csv(judge_output, mode="a", header=write_header, index=False)

    print(f"Done. Judged results saved to {judge_output!r}.")


# Uncomment to run:
# run_batch_judging()

## Cell Group 12 — Summary Statistics

In [ ]:
judged = pd.read_csv(JUDGE_OUTPUT_FILE)
n = len(judged)
print(f"Total judged pairs: {n:,}\n")

for col, label in [("rag_verdict", "RAG"), ("no_rag_verdict", "No-RAG")]:
    counts = judged[col].value_counts()
    unsafe_rate = counts.get("unsafe", 0) / n * 100
    print(f"{label} unsafe rate: {unsafe_rate:.1f}% ({counts.get('unsafe', 0):,}/{n:,})")

print()

# Unsafe category breakdown.
for col, cat_col, label in [
    ("rag_verdict",    "rag_category",    "RAG"),
    ("no_rag_verdict", "no_rag_category", "No-RAG"),
]:
    unsafe_rows = judged[judged[col] == "unsafe"]
    if not unsafe_rows.empty:
        print(f"{label} unsafe category breakdown:")
        print(unsafe_rows[cat_col].value_counts().to_string())
        print()

# Disagreement sample: safe with RAG, unsafe without (or vice versa).
rag_safe_no_rag_unsafe = judged[
    (judged["rag_verdict"] == "safe") & (judged["no_rag_verdict"] == "unsafe")
]
no_rag_safe_rag_unsafe = judged[
    (judged["rag_verdict"] == "unsafe") & (judged["no_rag_verdict"] == "safe")
]

print(f"Safe with RAG, unsafe without: {len(rag_safe_no_rag_unsafe):,}")
print(f"Unsafe with RAG, safe without: {len(no_rag_safe_rag_unsafe):,}")

if not rag_safe_no_rag_unsafe.empty:
    print("\nSample (safe RAG / unsafe no-RAG):")
    display(rag_safe_no_rag_unsafe.head(3)[["question", "rag_response", "no_rag_response", "no_rag_category"]])

if not no_rag_safe_rag_unsafe.empty:
    print("\nSample (unsafe RAG / safe no-RAG):")
    display(no_rag_safe_rag_unsafe.head(3)[["question", "rag_response", "no_rag_response", "rag_category"]])